## 03 Execution

Demonstrate simplified execution analytics using visible order book depth.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.execution_analysis import (
    generate_execution_windows,
    run_signal_adjusted_execution_comparison_for_window,
    run_multi_window_signal_adjusted_analysis,
    summarize_signal_adjusted_execution,
)

In [2]:
data_path = PROJECT_ROOT / "data" / "processed" / "obi_dataset.parquet"
market_df = pd.read_parquet(data_path).sort_values("timestamp").reset_index(drop=True)
market_df[["timestamp", "mid_price", "ask_p1", "ask_q1", "ask_depth_5", "obi_5"]].head()

,timestamp,mid_price,ask_p1,ask_q1,ask_depth_5,obi_5
0,2021-04-07 11:32:42.122161+00:00,56035.995,56036.0,1902.290039,113298.046913,0.175507
1,2021-04-07 11:32:43.122161+00:00,56035.995,56036.0,1902.290039,113298.046913,0.175507
2,2021-04-07 11:32:44.122161+00:00,56035.995,56036.0,1902.290039,113298.046913,0.175507
3,2021-04-07 11:32:45.122161+00:00,56035.995,56036.0,1902.290039,113298.046913,0.175507
4,2021-04-07 11:32:46.122161+00:00,56035.995,56036.0,1902.290039,113298.046913,0.175507


### Single Window

Compare TWAP, liquidity-weighted schedule, and OBI-adjusted TWAP for one rolling execution window.

In [3]:
windows = generate_execution_windows(market_df, window_minutes=30, step_minutes=30, max_windows=5)
start_time, end_time = windows[0]

metrics_df, details = run_signal_adjusted_execution_comparison_for_window(
    market_df,
    start_time=start_time,
    end_time=end_time,
    side="buy",
    target_qty=500_000,
    num_slices=30,
    fill_method="depth_walk",
    levels=5,
    use_regime_filter=True,
)

metrics_df[["strategy", "slippage_bps", "fill_rate", "avg_levels_used", "pct_child_orders_walked_beyond_top"]]

,strategy,slippage_bps,fill_rate,avg_levels_used,pct_child_orders_walked_beyond_top
0,twap,41.114219,0.519566,1.0,0.0
1,liquidity_weighted_schedule,41.090372,0.553358,1.0,0.0
2,obi_adjusted_twap,40.930454,0.427089,1.0,0.0


### Multi-Window Summary

Rolling windows reduce dependence on a single price path.

In [4]:
windows = generate_execution_windows(market_df, window_minutes=30, step_minutes=30, max_windows=20)
results = run_multi_window_signal_adjusted_analysis(
    market_df,
    windows=windows,
    side="buy",
    target_qty=500_000,
    num_slices=30,
    fill_method="depth_walk",
    levels=5,
    max_windows=20,
    use_regime_filter=True,
)
summary = summarize_signal_adjusted_execution(results, output_dir=PROJECT_ROOT / "reports" / "selected_tables")
summary

,strategy,mean_slippage_bps,median_slippage_bps,mean_execution_cost,median_execution_cost,mean_fill_rate,mean_avg_levels_used,mean_pct_walked_beyond_top,mean_requested_qty,mean_executed_qty,mean_unfilled_qty,mean_order_size_to_visible_depth_ratio,lowest_slippage_count,lowest_slippage_share
0,liquidity_weighted_schedule,1.506214,-5.523644,1.155226e+06,-9.685817e+06,0.746411,0.998333,0.0,500000.0,373205.655755,126794.344245,2.953203,5,0.25
1,obi_adjusted_twap,0.311081,-6.318706,-2.613414e+06,-1.041987e+07,0.641661,0.998333,0.0,500000.0,320830.291484,179169.708516,2.953203,13,0.65
2,twap,0.842235,-6.191396,-2.346430e+05,-1.059237e+07,0.684977,0.998333,0.0,500000.0,342488.329053,157511.670947,2.953203,2,0.10
